In [15]:
## Testing the music datasets
import pandas as pd

discogs = pd.read_xml("input/datasets/discogs.xml")
musicbrainz = pd.read_xml("input/datasets/musicbrainz.xml")
lastfm = pd.read_xml("input/datasets/lastfm.xml")

In [39]:
# Testing discogs <-> musicbrainz train/test sets
from PyDI.entitymatching import FeatureExtractor
import pandas as pd
from pathlib import Path
from PyDI.io import load_csv
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator



INPUT_DIR = Path("output/big/music4")

# Load ground truth correspondences
aa2a_train = load_csv(
    INPUT_DIR / "discogs_musicbrainz_goldstandard_blocking.csv",
    name="discogs_musicbrainz_goldstandard_blocking",
    header=0,
    add_index=False
)
# Debug: print actual columns
print(f"Train file columns: {aa2a_train.columns.tolist()}")
# Rename train columns to match expected format (id1, id2) if needed
if 'id_a' in aa2a_train.columns:
    aa2a_train = aa2a_train.rename(columns={'id_a': 'id1', 'id_b': 'id2'})

# Load test set - use pandas directly for proper CSV parsing
aa2a_test_raw = pd.read_csv(
    "input/datasets/musicbrainz_2_discogs_test.csv",
    header=None
)
print(f"Raw test file shape: {aa2a_test_raw.shape}")
print(f"Raw test file columns: {aa2a_test_raw.columns.tolist()}")
print(f"First 3 rows raw:\n{aa2a_test_raw.head(3)}")
print(f"Unique values in last column: {aa2a_test_raw.iloc[:, -1].unique()[:10] if aa2a_test_raw.shape[1] > 0 else 'N/A'}")

# Check if it's actually comma-separated or has different structure
if aa2a_test_raw.shape[1] == 1:
    # File might be using different delimiter or is malformed
    # Try splitting the single column
    print("Attempting to split single column...")
    aa2a_test = aa2a_test_raw.iloc[:, 0].str.split(',', expand=True)
    # File format is: mbrainz_id, discogs_id, label
    # But correspondences use: discogs_id as id1, mbrainz_id as id2
    # So we need to SWAP the columns to match
    aa2a_test.columns = ['id2', 'id1', 'label']  # Swap: mbrainz->id2, discogs->id1
else:
    aa2a_test = aa2a_test_raw.copy()
    aa2a_test.columns = ['id2', 'id1', 'label']  # Swap order

# Reorder columns to standard format
aa2a_test = aa2a_test[['id1', 'id2', 'label']]

# Convert label from TRUE/FALSE strings to 1/0
aa2a_test['label'] = aa2a_test['label'].str.upper().map({'TRUE': 1, 'FALSE': 0})
print(f"After label conversion - NaN count: {aa2a_test['label'].isna().sum()}")
aa2a_test = aa2a_test.dropna(subset=['label'])
aa2a_test['label'] = aa2a_test['label'].astype(int)

print(f"Train set: {len(aa2a_train)} pairs, {aa2a_train['label'].sum()} matches")
print(f"Test set: {len(aa2a_test)} pairs, {aa2a_test['label'].sum()} matches")
print(f"Train columns: {aa2a_train.columns.tolist()}")
print(f"Test columns: {aa2a_test.columns.tolist()}")
def normalize_country(df, col):
    country_map = {
        'uk': 'united kingdom',
        'united kingdom of great britain and northern ireland': 'united kingdom',
        'usa': 'united states',
        'united states of america': 'united states',
        'us': 'united states',
        'deutschland': 'germany',
        'de': 'germany',
        'ger': 'germany',
        'fr': 'france',
        'jp': 'japan',
        'au': 'australia',
        'ca': 'canada',
        'nl': 'netherlands',
        'holland': 'netherlands',
        'brasil': 'brazil',
        'espana': 'spain',
        'italia': 'italy',
    }
    
    def normalize(val):
        if pd.isna(val):
            return val
        v = str(val).strip().lower()
        return country_map.get(v, v)
    
    return df[col].apply(normalize)

# Apply before feature extraction
discogs['country_normalized'] = normalize_country(discogs, 'release-country')
musicbrainz['country_normalized'] = normalize_country(musicbrainz, 'release-country')


comparators = [
    # Release name — Jaccard
    StringComparator(
        column='name', 
        similarity_function='jaccard',
    ),
    # Release artist — Jaccard
    StringComparator(
        column='artist',
        similarity_function='jaccard',
    ),
    # Release duration — within 10% --> allow 10% deviation
    NumericComparator(
        column='duration',
        method='relative_difference',
        max_difference=0.10
    ),
    # Release date — within 2 years
    DateComparator(
        column='release-date',
        max_days_difference=365 * 2
    ),
    # Release country — Jaccard
    StringComparator(
        column='release-country',
        similarity_function='jaccard',
    ),
]

feature_extractor = FeatureExtractor(comparators)

# Extract features using FeatureExtractor
# aa2a_train already has id1, id2 columns
pairs_df = aa2a_train[['id1', 'id2']].copy()
train_features = feature_extractor.create_features(
    discogs, musicbrainz, pairs_df, labels=aa2a_train['label'], id_column='id'
)

print(f"✅ Training features extracted!")
print(f"Feature columns: {[col for col in train_features.columns if col not in ['id1', 'id2', 'label']]}")

# Prepare data for ML training
feature_columns = [col for col in train_features.columns if col not in ['id1', 'id2', 'label']]

X_train = train_features[feature_columns]
y_train = train_features['label']

print(f"Training data: X={X_train.shape}, y={y_train.shape}")
print(f"Class distribution: {y_train.value_counts().to_dict()}")

Train file columns: ['id_a', 'id_b', 'label']
Raw test file shape: (5805, 1)
Raw test file columns: [0]
First 3 rows raw:
                                   0
0   mbrainz_5809,discogs_71208,FALSE
1  mbrainz_5800,discogs_150354,FALSE
2   mbrainz_5800,discogs_72921,FALSE
Unique values in last column: ['mbrainz_5809,discogs_71208,FALSE' 'mbrainz_5800,discogs_150354,FALSE'
 'mbrainz_5800,discogs_72921,FALSE' 'mbrainz_8079,discogs_36293,TRUE'
 'mbrainz_17654,discogs_182489,FALSE' 'mbrainz_15000,discogs_59197,FALSE'
 'mbrainz_16332,discogs_81242,TRUE' 'mbrainz_16332,discogs_110881,FALSE'
 'mbrainz_13290,discogs_104973,FALSE' 'mbrainz_13290,discogs_35333,FALSE']
Attempting to split single column...
After label conversion - NaN count: 0
Train set: 300 pairs, 150 matches
Test set: 5805 pairs, 475 matches
Train columns: ['id1', 'id2', 'label']
Test columns: ['id1', 'id2', 'label']
✅ Training features extracted!
Feature columns: ['StringComparator(name, jaccard, tokenization=word, list_strategy=N

In [40]:
# Set up GridSearchCV with multiple models and hyperparameters
print(f"\n🔍 Setting up GridSearchCV...")

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score

# Define models and parameter grids
param_grids = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'class_weight': ['balanced', None]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l2'],
            'class_weight': ['balanced', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.1, 0.2],
            'max_depth': [3, 5],
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear'],
            'class_weight': ['balanced', None]
        }
    }
}

# Use F1 score as the scoring metric (good for imbalanced data)
scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"GridSearch setup: {len(param_grids)} models, F1 scoring, 5-fold CV")

# Train models using GridSearchCV
print(f"\n🚀 Training Models with GridSearchCV...")

grid_search_results = {}
best_overall_score = -1
best_overall_model = None
best_model_name = None

for model_name, config in param_grids.items():
    print(f"\nTraining {model_name}...")
    

    # Create GridSearchCV
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        scoring=scorer,
        cv=cv_folds,
        n_jobs=-1,  # Use all available cores
        verbose=0
    )
    
    # Fit GridSearchCV
    grid_search.fit(X_train, y_train)
    
    # Store results
    grid_search_results[model_name] = {
        'grid_search': grid_search,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    }
    
    print(f"  ✅ {model_name}: Best CV F1 = {grid_search.best_score_:.4f}")
    print(f"     Best params: {grid_search.best_params_}")
    
    # Track overall best model
    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        best_overall_model = grid_search.best_estimator_
        best_model_name = model_name
            
print(f"\n🏆 Best Overall Model: {best_model_name} (CV F1: {best_overall_score:.4f})")



🔍 Setting up GridSearchCV...
GridSearch setup: 4 models, F1 scoring, 5-fold CV

🚀 Training Models with GridSearchCV...

Training RandomForest...
  ✅ RandomForest: Best CV F1 = 0.9204
     Best params: {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}

Training LogisticRegression...
  ✅ LogisticRegression: Best CV F1 = 0.9347
     Best params: {'C': 1.0, 'class_weight': 'balanced', 'penalty': 'l2'}

Training GradientBoosting...


/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty'

  ✅ GradientBoosting: Best CV F1 = 0.9166
     Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}

Training SVM...
  ✅ SVM: Best CV F1 = 0.9289
     Best params: {'C': 0.1, 'class_weight': 'balanced', 'kernel': 'linear'}

🏆 Best Overall Model: LogisticRegression (CV F1: 0.9347)


In [41]:
from PyDI.entitymatching import MLBasedMatcher
from PyDI.entitymatching import TokenBlocker, EmbeddingBlocker
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator

embedding_blocker_aa2a = EmbeddingBlocker(
    df_left=discogs,
    df_right=musicbrainz,
    text_cols=['artist'],
    id_column='id',
    model='sentence-transformers/all-MiniLM-L6-v2',
    top_k=100,
    threshold=0.5
)
embedding_candidates_aa2a = embedding_blocker_aa2a.materialize()

# Create MLBasedMatcher and apply trained model
ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_aa2a = ml_matcher.match(
    discogs, musicbrainz, candidates=embedding_candidates_aa2a, id_column='id', trained_classifier=best_overall_model
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_aa2a,
    test_pairs=aa2a_test,
    out_dir=INPUT_DIR
)

display(eval_results)

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_aa2a,
    out_dir=INPUT_DIR / "cluster_analysis"
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

{'precision': 0.7872340425531915,
 'recall': 0.8568421052631578,
 'f1': 0.8205645161290323,
 'accuracy': 0.9693367786391042,
 'true_positives': 407,
 'false_positives': 110,
 'false_negatives': 68,
 'true_negatives': 5220,
 'threshold_used': 0.0,
 'total_correspondences': 3101,
 'filtered_correspondences': 3101,
 'evaluation_timestamp': '2026-02-01T17:04:08.354496',
 'output_files': ['output/big/music4/matching_evaluation_summary.json',
  'output/big/music4/matching_detailed_results.csv']}


📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,2258,87.519380
1,3,225,8.720930
2,4,61,2.364341
3,5,19,0.736434
4,6,12,0.465116
5,7,4,0.155039
6,21,1,0.038760


In [ ]:
# Load detailed results and actual record data to understand errors
detailed = pd.read_csv(INPUT_DIR / "matching_detailed_results.csv")

# Load the source datasets
discogs = pd.read_xml("input/datasets/discogs.xml")
musicbrainz = pd.read_xml("input/datasets/musicbrainz.xml")

# Debug: Check ID formats - they already have prefixes!
print("=== ID FORMAT DEBUGGING ===")
print(f"\nDetailed results - sample id1: {detailed['id1'].head(3).tolist()}")
print(f"Discogs 'id' column sample: {discogs['id'].head(3).tolist()}")
print(f"\nDetailed results - sample id2: {detailed['id2'].head(3).tolist()}")
print(f"MusicBrainz 'id' column sample: {musicbrainz['id'].head(3).tolist()}")

# IDs already have prefixes - match directly
# Create lookup dictionaries for fast access
discogs_dict = discogs.set_index('id').to_dict('index')
musicbrainz_dict = musicbrainz.set_index('id').to_dict('index')

print(f"\nDiscogs records loaded: {len(discogs_dict)}")
print(f"MusicBrainz records loaded: {len(musicbrainz_dict)}")

# Get false positives
false_positives = detailed[detailed['classification'] == 'FP'].copy()
print(f"\n❌ FALSE POSITIVES: {len(false_positives)} pairs incorrectly predicted as matches")

# Examine false positives with actual record content
print(f"\n🔍 Examining FALSE POSITIVES (model thought these match, but they don't):\n")
print("=" * 100)

for idx, row in false_positives.head(15).iterrows():
    discogs_id = row['id1']
    mbrainz_id = row['id2']
    score = row['score']
    
    print(f"\n📌 Pair #{idx} | Score: {score:.3f}")
    print("-" * 80)
    
    # Look up directly in dictionaries
    if discogs_id in discogs_dict:
        rec = discogs_dict[discogs_id]
        print(f"DISCOGS ({discogs_id}):")
        for col, val in rec.items():
            if pd.notna(val):
                val_str = str(val)[:80]
                print(f"   {col}: {val_str}")
    else:
        print(f"DISCOGS ({discogs_id}): NOT FOUND")
    
    print()
    
    if mbrainz_id in musicbrainz_dict:
        rec = musicbrainz_dict[mbrainz_id]
        print(f"MUSICBRAINZ ({mbrainz_id}):")
        for col, val in rec.items():
            if pd.notna(val):
                val_str = str(val)[:80]
                print(f"   {col}: {val_str}")
    else:
        print(f"MUSICBRAINZ ({mbrainz_id}): NOT FOUND")
    
    print("=" * 100)

# Check false negatives too
false_negatives = detailed[detailed['classification'] == 'FN'].copy()
if len(false_negatives) > 0:
    print(f"\n\n⚠️ FALSE NEGATIVES: {len(false_negatives)} true matches that were MISSED:\n")
    print("=" * 100)
    
    for idx, row in false_negatives.iterrows():
        discogs_id = row['id1']
        mbrainz_id = row['id2']
        score = row['score']
        
        print(f"\n📌 Missed Match | Score: {score:.3f}")
        print("-" * 80)
        
        if discogs_id in discogs_dict:
            rec = discogs_dict[discogs_id]
            print(f"DISCOGS ({discogs_id}):")
            for col, val in rec.items():
                if pd.notna(val):
                    print(f"   {col}: {str(val)[:80]}")
        
        print()
        
        if mbrainz_id in musicbrainz_dict:
            rec = musicbrainz_dict[mbrainz_id]
            print(f"MUSICBRAINZ ({mbrainz_id}):")
            for col, val in rec.items():
                if pd.notna(val):
                    print(f"   {col}: {str(val)[:80]}")
        
        print("=" * 100)

In [42]:
# Testing musicbrainz <-> lastfm train/test sets
from PyDI.entitymatching import FeatureExtractor
import pandas as pd
import re
from pathlib import Path
from PyDI.io import load_csv
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator

INPUT_DIR = Path("output/big/music4")

# ===== PREPROCESSING FUNCTIONS =====
def clean_album_name(text):
    """Clean album names - remove Last.fm prefixes and normalize."""
    if not text or pd.isna(text) or text == 'nan':
        return ""
    text = str(text).lower().strip()
    # Remove Last.fm prefixes
    text = re.sub(r'^\[explicit\]\s*', '', text)
    text = re.sub(r'^\*popular\*\s*', '', text)
    text = re.sub(r'^- new -\s*', '', text)
    text = re.sub(r'^album:\s*', '', text, flags=re.IGNORECASE)
    # Remove artist prefix pattern "Artist Name - " at start
    text = re.sub(r'^[\w\s\.\-&]+\s+-\s+', '', text)
    # Normalize quotes and punctuation
    text = re.sub(r"[''`]", "'", text)
    text = re.sub(r'["""]', '"', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def normalize_artist(text):
    """Normalize artist names - handle LastName, FirstName format and abbreviations."""
    if not text or pd.isna(text) or text == 'nan':
        return ""
    text = str(text).lower().strip()
    # Handle "LastName, FirstName" → "FirstName LastName"
    if ',' in text:
        parts = text.split(',', 1)
        if len(parts) == 2 and len(parts[1].strip()) > 0:
            text = f"{parts[1].strip()} {parts[0].strip()}"
    # Remove ", The" from end
    text = re.sub(r',\s*the$', '', text)
    # Remove "the " from start
    text = re.sub(r'^the\s+', '', text)
    # Normalize spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Load datasets
musicbrainz = pd.read_xml('input/datasets/musicbrainz.xml')
lastfm = pd.read_xml('input/datasets/lastfm.xml')

print(f"MusicBrainz: {len(musicbrainz)} records, cols: {musicbrainz.columns.tolist()}")
print(f"LastFM: {len(lastfm)} records, cols: {lastfm.columns.tolist()}")

# ===== APPLY PREPROCESSING =====
# Create cleaned columns
musicbrainz['name_clean'] = musicbrainz['name'].apply(clean_album_name)
musicbrainz['artist_clean'] = musicbrainz['artist'].apply(normalize_artist)
lastfm['name_clean'] = lastfm['name'].apply(clean_album_name)
lastfm['artist_clean'] = lastfm['artist'].apply(normalize_artist)

# Normalize numeric fields - convert to float, treat 0 as NaN for duration
musicbrainz['duration'] = pd.to_numeric(musicbrainz['duration'], errors='coerce')
lastfm['duration'] = pd.to_numeric(lastfm['duration'], errors='coerce')
# Replace 0 duration with NaN (missing data)
musicbrainz.loc[musicbrainz['duration'] == 0, 'duration'] = pd.NA
lastfm.loc[lastfm['duration'] == 0, 'duration'] = pd.NA

print(f"\n✅ Preprocessing applied!")
print(f"Sample MusicBrainz - name: '{musicbrainz['name'].iloc[0]}' → clean: '{musicbrainz['name_clean'].iloc[0]}'")
print(f"Sample LastFM - name: '{lastfm['name'].iloc[0]}' → clean: '{lastfm['name_clean'].iloc[0]}'")

# Load ground truth correspondences
aa2a_train = load_csv(
    INPUT_DIR / "musicbrainz_lastfm_goldstandard_blocking.csv",
    name="musicbrainz_lastfm_goldstandard_blocking",
    header=0,
    add_index=False
)
print(f"\nTrain file columns: {aa2a_train.columns.tolist()}")

# Rename train columns to match expected format (id1, id2)
if 'id_a' in aa2a_train.columns:
    aa2a_train = aa2a_train.rename(columns={'id_a': 'id1', 'id_b': 'id2'})

# Load test set
aa2a_test_raw = pd.read_csv(
    "input/datasets/musicbrainz_2_lastfm_test.csv",
    header=None,
    dtype=str
)
print(f"Raw test file shape: {aa2a_test_raw.shape}")

# Test file has 3 columns: mbrainz_id, lastfm_id, label
aa2a_test = aa2a_test_raw.copy()
aa2a_test.columns = ['id1', 'id2', 'label']
aa2a_test['id1'] = aa2a_test['id1'].astype(str).str.strip()
aa2a_test['id2'] = aa2a_test['id2'].astype(str).str.strip()

# Convert label from true/false strings to 1/0
aa2a_test['label'] = aa2a_test['label'].astype(str).str.strip().str.upper().map({'TRUE': 1, 'FALSE': 0})
aa2a_test = aa2a_test.dropna(subset=['label'])
aa2a_test['label'] = aa2a_test['label'].astype(int)

print(f"Train: {len(aa2a_train)} pairs, {aa2a_train['label'].sum()} matches")
print(f"Test: {len(aa2a_test)} pairs, {aa2a_test['label'].sum()} matches")

# ===== IMPROVED COMPARATORS WITH CLEANED COLUMNS =====
similarity_comparators = [
    # ===== CLEANED NAME (most important) =====
    StringComparator("name_clean", similarity_function="jaro_winkler"),
    StringComparator("name_clean", similarity_function="levenshtein"),
    
    
    # ===== ORIGINAL NAME (backup - catches edge cases) =====
    StringComparator("name", similarity_function="jaro_winkler", preprocess=str.lower),
    
    # ===== CLEANED ARTIST (critical for disambiguation) =====
    StringComparator("artist_clean", similarity_function="jaro_winkler"),
    StringComparator("artist_clean", similarity_function="levenshtein"),
    
    
]

feature_extractor = FeatureExtractor(similarity_comparators)

# Extract features
pairs_df = aa2a_train[['id1', 'id2']].copy()
train_features = feature_extractor.create_features(
    musicbrainz, lastfm, pairs_df, labels=aa2a_train['label'], id_column='id'
)

print(f"\n✅ Training features extracted!")
feature_columns = [col for col in train_features.columns if col not in ['id1', 'id2', 'label']]
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")

# Prepare data for ML training
X_train = train_features[feature_columns]
y_train = train_features['label']

print(f"\nTraining data: X={X_train.shape}, y={y_train.shape}")
print(f"Class distribution: {y_train.value_counts().to_dict()}")

# Show feature statistics
print(f"\n📊 Feature statistics:")
print(X_train.describe().round(3).T[['mean', 'std', 'min', 'max']])

MusicBrainz: 4763 records, cols: ['id', 'name', 'artist', 'release-date', 'release-country', 'duration', 'tracks']
LastFM: 9865 records, cols: ['id', 'name', 'artist', 'duration', 'tracks']

✅ Preprocessing applied!
Sample MusicBrainz - name: 'Fermats Theorem / Sight Beyond' → clean: 'fermats theorem / sight beyond'
Sample LastFM - name: 'John B -  Fermats Theorem / Sight Beyond' → clean: 'fermats theorem / sight beyond'

Train file columns: ['id_a', 'id_b', 'label']
Raw test file shape: (5249, 3)
Train: 300 pairs, 150 matches
Test: 5249 pairs, 517 matches

✅ Training features extracted!
Feature columns (5): ['StringComparator(name_clean, jaro_winkler, tokenization=char, list_strategy=None)', 'StringComparator(name_clean, levenshtein, tokenization=char, list_strategy=None)', 'StringComparator(name, jaro_winkler, tokenization=char, list_strategy=None)', 'StringComparator(artist_clean, jaro_winkler, tokenization=char, list_strategy=None)', 'StringComparator(artist_clean, levenshtein, tok

In [43]:
# GridSearchCV for musicbrainz <-> lastfm
print(f"\n🔍 Setting up GridSearchCV...")

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score

param_grids = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'class_weight': ['balanced', None]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l2'],
            'class_weight': ['balanced', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.1, 0.2],
            'max_depth': [3, 5],
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear'],
            'class_weight': ['balanced', None]
        }
    }
}

scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"GridSearch setup: {len(param_grids)} models, F1 scoring, 5-fold CV")
print(f"\n🚀 Training Models with GridSearchCV...")

grid_search_results = {}
best_overall_score = -1
best_overall_model = None
best_model_name = None

for model_name, config in param_grids.items():
    print(f"\nTraining {model_name}...")
    
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        scoring=scorer,
        cv=cv_folds,
        n_jobs=-1,
        verbose=0
    )
    
    grid_search.fit(X_train, y_train)
    
    grid_search_results[model_name] = {
        'grid_search': grid_search,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    }
    
    print(f"  ✅ {model_name}: Best CV F1 = {grid_search.best_score_:.4f}")
    print(f"     Best params: {grid_search.best_params_}")
    
    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        best_overall_model = grid_search.best_estimator_
        best_model_name = model_name
            
print(f"\n🏆 Best Overall Model: {best_model_name} (CV F1: {best_overall_score:.4f})")


🔍 Setting up GridSearchCV...
GridSearch setup: 4 models, F1 scoring, 5-fold CV

🚀 Training Models with GridSearchCV...

Training RandomForest...
  ✅ RandomForest: Best CV F1 = 0.8695
     Best params: {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 50}

Training LogisticRegression...
  ✅ LogisticRegression: Best CV F1 = 0.8728
     Best params: {'C': 10.0, 'class_weight': 'balanced', 'penalty': 'l2'}

Training GradientBoosting...


/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty'

  ✅ GradientBoosting: Best CV F1 = 0.8701
     Best params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 50}

Training SVM...
  ✅ SVM: Best CV F1 = 0.8746
     Best params: {'C': 1.0, 'class_weight': 'balanced', 'kernel': 'linear'}

🏆 Best Overall Model: SVM (CV F1: 0.8746)


In [44]:
# Evaluation for musicbrainz <-> lastfm
print("\n🧪 Evaluating on musicbrainz <-> lastfm test set...")

from PyDI.entitymatching import MLBasedMatcher, EmbeddingBlocker
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator

# Use EmbeddingBlocker with correct API
embedding_blocker = EmbeddingBlocker(
    df_left=musicbrainz,
    df_right=lastfm,
    text_cols=['name', 'artist'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=300,
    batch_size=500,
    output_dir=INPUT_DIR / "blocking-evaluation-music",
    id_column='id'
)
candidate_pairs = embedding_blocker.materialize()
print(f"Candidate pairs from blocking: {len(candidate_pairs)}")

# Apply ML-based matching with best model
ml_matcher = MLBasedMatcher(feature_extractor)

matches = ml_matcher.match(
    musicbrainz, lastfm, 
    candidates=candidate_pairs, 
    id_column='id', 
    trained_classifier=best_overall_model,
    threshold=0.2
)
print(f"Matches found: {len(matches)}")

# Evaluate using the correct API
eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=matches,
    test_pairs=aa2a_test,
    out_dir=INPUT_DIR
)

print(f"\n📊 Evaluation Results:")
display(eval_results)


🧪 Evaluating on musicbrainz <-> lastfm test set...
Candidate pairs from blocking: 1066606
Matches found: 2301

📊 Evaluation Results:


{'precision': 0.9783783783783784,
 'recall': 0.7001934235976789,
 'f1': 0.8162344983089064,
 'accuracy': 0.9689464659935225,
 'true_positives': 362,
 'false_positives': 8,
 'false_negatives': 155,
 'true_negatives': 4724,
 'threshold_used': 0.0,
 'total_correspondences': 2301,
 'filtered_correspondences': 2301,
 'evaluation_timestamp': '2026-02-01T17:07:02.860735',
 'output_files': ['output/big/music4/matching_evaluation_summary.json',
  'output/big/music4/matching_detailed_results.csv']}

In [33]:
# Analyze false negatives - why are we missing 392 matches?
print("🔍 Analyzing False Negatives...")

# Get all true matches from test set
true_matches = aa2a_test[aa2a_test['label'] == 1][['id1', 'id2']].copy()
print(f"Total true matches in test set: {len(true_matches)}")

# Get our predicted matches
predicted_set = set(zip(matches['id1'], matches['id2']))

# Check if missed matches were in candidate pairs
candidate_set = set(zip(candidate_pairs['id1'], candidate_pairs['id2']))
true_matches['in_candidates'] = true_matches.apply(
    lambda r: (r['id1'], r['id2']) in candidate_set, axis=1
)
true_matches['was_found'] = true_matches.apply(
    lambda r: (r['id1'], r['id2']) in predicted_set, axis=1
)

missed_in_candidates = (~true_matches['was_found'] & true_matches['in_candidates']).sum()
missed_not_in_candidates = (~true_matches['was_found'] & ~true_matches['in_candidates']).sum()

print(f"\n📊 False Negative Breakdown:")
print(f"  - Missed but IN candidates (model rejected): {missed_in_candidates}")
print(f"  - Missed and NOT in candidates (blocking issue): {missed_not_in_candidates}")
print(f"\n🎯 Blocking Recall: {true_matches['in_candidates'].mean():.2%}")

🔍 Analyzing False Negatives...
Total true matches in test set: 517

📊 False Negative Breakdown:
  - Missed but IN candidates (model rejected): 377
  - Missed and NOT in candidates (blocking issue): 0

🎯 Blocking Recall: 100.00%


In [34]:
# Analyze False Negatives - Why are we missing 442 matches?
print("🔍 Analyzing False Negatives...\n")

# Get all true matches from test set
true_match_pairs = aa2a_test[aa2a_test['label'] == 1][['id1', 'id2']].copy()
print(f"Total true matches in test set: {len(true_match_pairs)}")

# Get predicted matches
predicted_set = set(zip(matches['id1'], matches['id2']))
print(f"Our predictions: {len(predicted_set)}")

# Check if each true match was found
true_match_pairs['was_predicted'] = true_match_pairs.apply(
    lambda r: (r['id1'], r['id2']) in predicted_set, axis=1
)

# Check if each true match was in blocking candidates
candidate_set = set(zip(candidate_pairs['id1'], candidate_pairs['id2']))
true_match_pairs['in_candidates'] = true_match_pairs.apply(
    lambda r: (r['id1'], r['id2']) in candidate_set, axis=1
)

# Categorize false negatives
fn_in_candidates = true_match_pairs[~true_match_pairs['was_predicted'] & true_match_pairs['in_candidates']]
fn_not_in_candidates = true_match_pairs[~true_match_pairs['was_predicted'] & ~true_match_pairs['in_candidates']]

print(f"\n📊 False Negative Breakdown:")
print(f"  ✓ Found (True Positives): {true_match_pairs['was_predicted'].sum()}")
print(f"  ✗ Missed in candidates (model rejected): {len(fn_in_candidates)}")
print(f"  ✗ Missed NOT in candidates (blocking lost): {len(fn_not_in_candidates)}")
print(f"\n🎯 Blocking Recall: {true_match_pairs['in_candidates'].mean():.2%}")

# Show sample false negatives that were in candidates (model rejected them)
if len(fn_in_candidates) > 0:
    print(f"\n" + "="*70)
    print(f"📋 SAMPLE: Model Rejected True Matches ({len(fn_in_candidates)} total)")
    print("="*70)
    for i, (_, row) in enumerate(fn_in_candidates.head(10).iterrows()):
        mb = musicbrainz[musicbrainz['id'] == row['id1']]
        lf = lastfm[lastfm['id'] == row['id2']]
        if len(mb) > 0 and len(lf) > 0:
            mb_name = str(mb['name'].values[0])[:45] if pd.notna(mb['name'].values[0]) else "N/A"
            lf_name = str(lf['name'].values[0])[:45] if pd.notna(lf['name'].values[0]) else "N/A"
            mb_artist = str(mb['artist'].values[0])[:25] if pd.notna(mb['artist'].values[0]) else "N/A"
            lf_artist = str(lf['artist'].values[0])[:25] if pd.notna(lf['artist'].values[0]) else "N/A"
            print(f"\n{i+1}. {row['id1']} <-> {row['id2']}")
            print(f"   MB: '{mb_name}' by '{mb_artist}'")
            print(f"   LF: '{lf_name}' by '{lf_artist}'")

# Show sample false negatives lost by blocking
if len(fn_not_in_candidates) > 0:
    print(f"\n" + "="*70)
    print(f"📋 SAMPLE: Blocking Lost True Matches ({len(fn_not_in_candidates)} total)")
    print("="*70)
    for i, (_, row) in enumerate(fn_not_in_candidates.head(10).iterrows()):
        mb = musicbrainz[musicbrainz['id'] == row['id1']]
        lf = lastfm[lastfm['id'] == row['id2']]
        if len(mb) > 0 and len(lf) > 0:
            mb_name = str(mb['name'].values[0])[:45] if pd.notna(mb['name'].values[0]) else "N/A"
            lf_name = str(lf['name'].values[0])[:45] if pd.notna(lf['name'].values[0]) else "N/A"
            mb_artist = str(mb['artist'].values[0])[:25] if pd.notna(mb['artist'].values[0]) else "N/A"
            lf_artist = str(lf['artist'].values[0])[:25] if pd.notna(lf['artist'].values[0]) else "N/A"
            print(f"\n{i+1}. {row['id1']} <-> {row['id2']}")
            print(f"   MB: '{mb_name}' by '{mb_artist}'")
            print(f"   LF: '{lf_name}' by '{lf_artist}'")

🔍 Analyzing False Negatives...

Total true matches in test set: 517
Our predictions: 969

📊 False Negative Breakdown:
  ✓ Found (True Positives): 140
  ✗ Missed in candidates (model rejected): 377
  ✗ Missed NOT in candidates (blocking lost): 0

🎯 Blocking Recall: 100.00%

📋 SAMPLE: Model Rejected True Matches (377 total)

1. mbrainz_8031 <-> lastFM_17138
   MB: 'Bring It On' by 'New Cool Collective'
   LF: 'Bring It On' by 'N. Cool Collective'

2. mbrainz_25231 <-> lastFM_63658
   MB: 'Live at Spirit Square' by 'Jones, Marti'
   LF: '[explicit] Live at Spirit Square' by 'M. Jones'

3. mbrainz_1483 <-> lastFM_3458
   MB: 'Learning to Walk' by 'Sole'
   LF: 'Learning To Walk' by 'Sole'

4. mbrainz_10107 <-> lastFM_21535
   MB: 'Antidotes: Tour Edition' by 'Foals'
   LF: 'Antidotes:  Edition' by 'Foals'

5. mbrainz_7677 <-> lastFM_16401
   MB: 'Electronic Collection No. 1 (release)' by 'I Am Jen'
   LF: 'Electronic Collection No. 1' by 'I Am Jen'

6. mbrainz_18047 <-> lastFM_43511
   MB:

In [28]:
# Find optimal threshold - model is rejecting 392 true matches
print("🔧 Finding optimal threshold...")

# Get prediction probabilities for all candidates
X_candidates = feature_extractor.create_features(
    musicbrainz, lastfm, candidate_pairs[['id1', 'id2']], id_column='id'
)
feature_cols = [c for c in X_candidates.columns if c not in ['id1', 'id2', 'label']]

probs = best_overall_model.predict_proba(X_candidates[feature_cols])[:, 1]
X_candidates['prob'] = probs

# Merge with test labels
X_with_labels = X_candidates.merge(aa2a_test[['id1', 'id2', 'label']], on=['id1', 'id2'], how='inner')

# Test different thresholds
from sklearn.metrics import precision_score, recall_score, f1_score

print("\nThreshold | Precision | Recall | F1     | Matches")
print("-" * 55)
for thresh in [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1, 0.05]:
    pred = (X_with_labels['prob'] >= thresh).astype(int)
    if pred.sum() > 0:
        p = precision_score(X_with_labels['label'], pred)
        r = recall_score(X_with_labels['label'], pred)
        f1 = f1_score(X_with_labels['label'], pred)
        print(f"  {thresh:.2f}    |   {p:.3f}   |  {r:.3f} | {f1:.3f}  | {pred.sum()}")

🔧 Finding optimal threshold...


KeyboardInterrupt: 

1. TESTING GAMES DATASETS

In [18]:
## Testing sales <-> metacritic train/test sets
from PyDI.entitymatching import FeatureExtractor
import pandas as pd
from pathlib import Path
from PyDI.io import load_csv
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator

INPUT_DIR = Path("output/try/games3")

sales = pd.read_xml('input/datasets/sales.xml')
metacritic = pd.read_xml('input/datasets/metacritic.xml')

# Load ground truth correspondences
aa2a_train = load_csv(
    INPUT_DIR / "metacritic_sales_goldstandard_blocking.csv",
    name="metacritic_sales_goldstandard_blocking",
    header=1,
    names=['id_a', 'id_b', 'label'],
    add_index=False
)

aa2a_test = load_csv(
    INPUT_DIR / "metacritic_2_sales_test.csv",
    name="metacritic_2_sales_test.csv",
    header=None,
    names=['id_a', 'id_b', 'label'],
    add_index=False
)

# Parse test if single comma-separated column
test_raw = pd.read_csv(INPUT_DIR / "metacritic_2_sales_test.csv", dtype=str, header=None)
if test_raw.shape[1] == 1:
    split_test = test_raw.iloc[:, 0].str.split(',', expand=True)
    split_test.columns = ['id_a', 'id_b', 'label']
    aa2a_test = split_test.copy()

# Normalize to id1/id2 and swap to match TRAIN (sales -> metacritic)
aa2a_test = aa2a_test.rename(columns={'id_a': 'id1', 'id_b': 'id2'})
aa2a_test['id1'] = aa2a_test['id1'].astype(str).str.strip()
aa2a_test['id2'] = aa2a_test['id2'].astype(str).str.strip()

# If id1 looks like metacritic_ and id2 looks like sales_, swap
if aa2a_test['id1'].str.startswith('metacritic_').mean() > 0.5 and aa2a_test['id2'].str.startswith('sales_').mean() > 0.5:
    aa2a_test = aa2a_test.rename(columns={'id1': 'id2', 'id2': 'id1'})
aa2a_test = aa2a_test[['id1', 'id2', 'label']]

# Strip prefixes to match dataset ids if needed
def strip_prefix(val: str) -> str:
    return val.split('_', 1)[1] if isinstance(val, str) and '_' in val else val

aa2a_train['id_a'] = aa2a_train['id_a'].astype(str).str.strip().apply(strip_prefix)
aa2a_train['id_b'] = aa2a_train['id_b'].astype(str).str.strip().apply(strip_prefix)
aa2a_test['id1'] = aa2a_test['id1'].apply(strip_prefix)
aa2a_test['id2'] = aa2a_test['id2'].apply(strip_prefix)

# Normalize dataset ids
if 'id' in sales.columns:
    sales['id'] = sales['id'].astype(str).str.strip()
else:
    sales['id'] = sales.index.astype(str)
if 'id' in metacritic.columns:
    metacritic['id'] = metacritic['id'].astype(str).str.strip()
else:
    metacritic['id'] = metacritic.index.astype(str)

# If overlap is still near zero, fall back to index-based ids
sales_overlap = aa2a_test['id1'].isin(set(sales['id'])).mean()
meta_overlap = aa2a_test['id2'].isin(set(metacritic['id'])).mean()
if sales_overlap < 0.01 and meta_overlap < 0.01:
    sales['id'] = (sales.index + 1).astype(str)
    metacritic['id'] = (metacritic.index + 1).astype(str)
    print("Using index-based ids for sales/metacritic")

print("ID overlap check:")
print("  id1 in sales:", aa2a_test['id1'].isin(set(sales['id'])).mean())
print("  id2 in metacritic:", aa2a_test['id2'].isin(set(metacritic['id'])).mean())

similarity_comparators = [
    StringComparator("name", similarity_function="jaro_winkler", preprocess=str.lower),
    StringComparator("name", similarity_function="levenshtein", preprocess=str.lower),
    StringComparator("name", similarity_function="cosine", preprocess=str.lower),
    StringComparator("name", similarity_function="jaccard", preprocess=str.lower),
    DateComparator("releaseYear", max_days_difference=365),
    StringComparator("platform", similarity_function="jaccard", preprocess=str.lower, list_strategy='concatenate'),
    StringComparator("platform", similarity_function="jaccard", preprocess=str.lower, list_strategy='best_match'),
]

feature_extractor = FeatureExtractor(similarity_comparators)

pairs_df = aa2a_train[['id_a', 'id_b']].rename(columns={'id_a': 'id1', 'id_b': 'id2'})
train_features = feature_extractor.create_features(
    sales, metacritic, pairs_df, labels=aa2a_train['label'], id_column='id'
)

print("✅ Training features extracted!")
print(f"Feature columns: {[col for col in train_features.columns if col not in ['id1', 'id2', 'label']]}")

feature_columns = [col for col in train_features.columns if col not in ['id1', 'id2', 'label']]
X_train = train_features[feature_columns]
y_train = train_features['label']
print(f"Training data: X={X_train.shape}, y={y_train.shape}")
print(f"Class distribution: {y_train.value_counts().to_dict()}")

Using index-based ids for sales/metacritic
ID overlap check:
  id1 in sales: 0.9982847341337907
  id2 in metacritic: 0.9982847341337907
✅ Training features extracted!
Feature columns: ['StringComparator(name, jaro_winkler, tokenization=char, list_strategy=None)', 'StringComparator(name, levenshtein, tokenization=char, list_strategy=None)', 'StringComparator(name, cosine, tokenization=word, list_strategy=None)', 'StringComparator(name, jaccard, tokenization=word, list_strategy=None)', 'DateComparator(releaseYear, list_strategy=None)', 'StringComparator(platform, jaccard, tokenization=word, list_strategy=concatenate)', 'StringComparator(platform, jaccard, tokenization=word, list_strategy=best_match)']
Training data: X=(99, 7), y=(99,)
Class distribution: {0: 80, 1: 19}


In [19]:
# Set up GridSearchCV with multiple models and hyperparameters
print(f"\n🔍 Setting up GridSearchCV...")

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score

# Define models and parameter grids
param_grids = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'class_weight': ['balanced', None]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l2'],
            'class_weight': ['balanced', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.1, 0.2],
            'max_depth': [3, 5],
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear'],
            'class_weight': ['balanced', None]
        }
    }
}

# Use F1 score as the scoring metric (good for imbalanced data)
scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"GridSearch setup: {len(param_grids)} models, F1 scoring, 5-fold CV")

# Train models using GridSearchCV
print(f"\n🚀 Training Models with GridSearchCV...")

grid_search_results = {}
best_overall_score = -1
best_overall_model = None
best_model_name = None

for model_name, config in param_grids.items():
    print(f"\nTraining {model_name}...")
    

    # Create GridSearchCV
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        scoring=scorer,
        cv=cv_folds,
        n_jobs=-1,  # Use all available cores
        verbose=0
    )
    
    # Fit GridSearchCV
    grid_search.fit(X_train, y_train)
    
    # Store results
    grid_search_results[model_name] = {
        'grid_search': grid_search,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    }
    
    print(f"  ✅ {model_name}: Best CV F1 = {grid_search.best_score_:.4f}")
    print(f"     Best params: {grid_search.best_params_}")
    
    # Track overall best model
    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        best_overall_model = grid_search.best_estimator_
        best_model_name = model_name
            
print(f"\n🏆 Best Overall Model: {best_model_name} (CV F1: {best_overall_score:.4f})")



🔍 Setting up GridSearchCV...
GridSearch setup: 4 models, F1 scoring, 5-fold CV

🚀 Training Models with GridSearchCV...

Training RandomForest...
  ✅ RandomForest: Best CV F1 = 0.9143
     Best params: {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}

Training LogisticRegression...
  ✅ LogisticRegression: Best CV F1 = 0.8929
     Best params: {'C': 10.0, 'class_weight': None, 'penalty': 'l2'}

Training GradientBoosting...
  ✅ GradientBoosting: Best CV F1 = 0.9143
     Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}

Training SVM...


/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty'

  ✅ SVM: Best CV F1 = 0.9143
     Best params: {'C': 1.0, 'class_weight': 'balanced', 'kernel': 'rbf'}

🏆 Best Overall Model: RandomForest (CV F1: 0.9143)


In [21]:
# evaluation
from PyDI.entitymatching import MLBasedMatcher
from PyDI.entitymatching import EmbeddingBlocker
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator

embedding_blocker_aa2a = EmbeddingBlocker(
    df_left=sales,
    df_right=metacritic,
    text_cols=['name'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=500,
    output_dir=INPUT_DIR / "blocking-evaluation",
    id_column='id'
)
embedding_candidates_aa2a = embedding_blocker_aa2a.materialize()

# Create MLBasedMatcher and apply trained model
ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_aa2a = ml_matcher.match(
    sales, metacritic, candidates=embedding_candidates_aa2a, id_column='id', trained_classifier=best_overall_model
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_aa2a,
    test_pairs=aa2a_test,
    out_dir=INPUT_DIR
)

display(eval_results)

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_aa2a,
    out_dir=INPUT_DIR / "cluster_analysis"
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)


{'precision': 0.9734513274336283,
 'recall': 0.8148148148148148,
 'f1': 0.8870967741935484,
 'accuracy': 0.9518900343642611,
 'true_positives': 110,
 'false_positives': 3,
 'false_negatives': 25,
 'true_negatives': 444,
 'threshold_used': 0.0,
 'total_correspondences': 77156,
 'filtered_correspondences': 77156,
 'evaluation_timestamp': '2026-01-31T14:37:55.719275',
 'output_files': ['output/try/games3/matching_evaluation_summary.json',
  'output/try/games3/matching_detailed_results.csv']}


📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,34,91.891892
1,3,2,5.405405
2,18256,1,2.702703


In [14]:
## Testing metacritic dbpedia_games train/test sets
from PyDI.entitymatching import FeatureExtractor
import pandas as pd
from pathlib import Path
from PyDI.io import load_csv
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator

INPUT_DIR = Path("output/try/score/games4")

metacritic = pd.read_xml('input/datasets/metacritic_with_ids.xml')
dbpedia_games = pd.read_xml('input/datasets/dbpedia_games_with_ids.xml')

# Schema alignment - metacritic has: name, releaseYear; dbpedia_games has: gameLabel, releaseDate
dbpedia_games['name'] = dbpedia_games['gameLabel'].astype(str)
dbpedia_games['releaseYear'] = pd.to_datetime(dbpedia_games['releaseDate'], errors='coerce').dt.year

# Normalize metacritic releaseYear (it's stored as date string like "2018-01-01")
metacritic['releaseYear'] = pd.to_datetime(metacritic['releaseYear'], errors='coerce').dt.year

# Platform normalization function
def normalize_platform(val):
    if pd.isna(val):
        return val
    v = str(val).strip().lower()
    mapping = {
        'playstation 4': 'ps4', 'playstation4': 'ps4', 'ps 4': 'ps4',
        'playstation 3': 'ps3', 'playstation3': 'ps3', 'ps 3': 'ps3',
        'playstation 2': 'ps2', 'playstation2': 'ps2', 'ps 2': 'ps2',
        'playstation 5': 'ps5', 'playstation5': 'ps5', 'ps 5': 'ps5',
        'xbox one': 'xboxone', 'xbone': 'xboxone',
        'xbox 360': 'xbox360', 'x360': 'xbox360',
        'nintendo switch': 'switch', 'nintendo wii': 'wii', 'wii u': 'wiiu',
        'microsoft windows': 'pc', 'windows': 'pc', 'pc': 'pc',
        'game boy advance': 'gba', 'game boy color': 'gbc', 'game boy': 'gb',
        'playstation portable': 'psp', 'playstation vita': 'vita', 'psvita': 'vita',
    }
    return mapping.get(v, v)

metacritic['platform_normalized'] = metacritic['platform'].apply(normalize_platform)
dbpedia_games['platform_normalized'] = dbpedia_games['platform'].apply(normalize_platform)

# Normalize IDs
metacritic['id'] = metacritic['id'].astype(str).str.strip()
dbpedia_games['id'] = dbpedia_games['id'].astype(str).str.strip()

print(f"Metacritic: {len(metacritic)} records, cols: {metacritic.columns.tolist()}")
print(f"DBpedia Games: {len(dbpedia_games)} records, cols: {dbpedia_games.columns.tolist()}")

# Load ground truth correspondences
aa2a_train = load_csv(
    INPUT_DIR / "metacritic_dbpedia_games_goldstandard_blocking.csv",
    name="metacritic_dpedia_games_goldstandard_blocking",
    header=1,
    names=['id_a', 'id_b', 'label'],
    add_index=False
)

# Load test set - use pandas directly for proper CSV parsing (quoted single-column format)
aa2a_test_raw = pd.read_csv(
    "input/datasets/metacritic_2_dbpedia_test.csv",
    header=None,
    dtype=str
)
print(f"Raw test file shape: {aa2a_test_raw.shape}")

# File is single-column with quoted values - split it
if aa2a_test_raw.shape[1] == 1:
    aa2a_test = aa2a_test_raw.iloc[:, 0].str.split(',', expand=True)
    aa2a_test.columns = ['id_a', 'id_b', 'label']
else:
    aa2a_test = aa2a_test_raw.copy()
    aa2a_test.columns = ['id_a', 'id_b', 'label']

# Normalize test set IDs (dbpedia_ -> dbpedia_games_)
aa2a_test['id_a'] = aa2a_test['id_a'].astype(str).str.strip()
aa2a_test['id_b'] = aa2a_test['id_b'].astype(str).str.strip()
aa2a_test['id_b'] = aa2a_test['id_b'].str.replace(r'^dbpedia_(?!games_)', 'dbpedia_games_', regex=True)

# Convert label from TRUE/FALSE strings to 1/0
aa2a_test['label'] = aa2a_test['label'].astype(str).str.strip().str.upper().map({'TRUE': 1, 'FALSE': 0})
print(f"After label conversion - NaN count: {aa2a_test['label'].isna().sum()}")
aa2a_test = aa2a_test.dropna(subset=['label'])
aa2a_test['label'] = aa2a_test['label'].astype(int)

# Rename to standard format
aa2a_test = aa2a_test.rename(columns={'id_a': 'id1', 'id_b': 'id2'})
aa2a_test = aa2a_test[['id1', 'id2', 'label']]

print(f"Train: {len(aa2a_train)} pairs | Test: {len(aa2a_test)} pairs")
print(f"Test matches: {aa2a_test['label'].sum()} | Test non-matches: {(aa2a_test['label'] == 0).sum()}")

# IMPROVED similarity comparators
similarity_comparators = [
    # Name - multiple string similarity functions (most important feature)
    StringComparator("name", similarity_function="jaro_winkler", preprocess=str.lower),
    StringComparator("name", similarity_function="levenshtein", preprocess=str.lower),
    StringComparator("name", similarity_function="cosine", preprocess=str.lower),
    StringComparator("name", similarity_function="jaccard", preprocess=str.lower),
    
    # FIX: Use NumericComparator instead of DateComparator for year values
    NumericComparator("releaseYear"),
    
    # Platform - use normalized column with multiple strategies
    StringComparator("platform_normalized", similarity_function="jaccard", preprocess=str.lower, list_strategy='concatenate'),
    StringComparator("platform_normalized", similarity_function="jaccard", preprocess=str.lower, list_strategy='best_match'),
    
    # NEW: Add developer comparison (available in both datasets!)
    StringComparator("developer", similarity_function="jaro_winkler", preprocess=str.lower),
    StringComparator("developer", similarity_function="jaccard", preprocess=str.lower, list_strategy='best_match'),
]

feature_extractor = FeatureExtractor(similarity_comparators)

pairs_df = aa2a_train[['id_a', 'id_b']].rename(columns={'id_a': 'id1', 'id_b': 'id2'})
train_features = feature_extractor.create_features(
    metacritic, dbpedia_games, pairs_df, labels=aa2a_train['label'], id_column='id'
)

print("✅ Training features extracted!")
print(f"Feature columns: {[col for col in train_features.columns if col not in ['id1', 'id2', 'label']]}")

feature_columns = [col for col in train_features.columns if col not in ['id1', 'id2', 'label']]
X_train = train_features[feature_columns]
y_train = train_features['label']
print(f"Training data: X={X_train.shape}, y={y_train.shape}")
print(f"Class distribution: {y_train.value_counts().to_dict()}")

Metacritic: 20494 records, cols: ['id', 'name', 'releaseYear', 'developer', 'genres', 'platform', 'criticScore', 'userScore', 'ESRB', 'platform_normalized']
DBpedia Games: 65000 records, cols: ['gameLabel', 'platform', 'developer', 'genre', 'releaseDate', 'series', 'id', 'name', 'releaseYear', 'platform_normalized']
Raw test file shape: (579, 1)
After label conversion - NaN count: 0
Train: 99 pairs | Test: 579 pairs
Test matches: 187 | Test non-matches: 392
✅ Training features extracted!
Feature columns: ['StringComparator(name, jaro_winkler, tokenization=char, list_strategy=None)', 'StringComparator(name, levenshtein, tokenization=char, list_strategy=None)', 'StringComparator(name, cosine, tokenization=word, list_strategy=None)', 'StringComparator(name, jaccard, tokenization=word, list_strategy=None)', 'NumericComparator(releaseYear, absolute_difference, list_strategy=None)', 'StringComparator(platform_normalized, jaccard, tokenization=word, list_strategy=concatenate)', 'StringCompara

In [15]:
# Set up GridSearchCV with multiple models and hyperparameters
print(f"\n🔍 Setting up GridSearchCV...")

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score

# Define models and parameter grids
param_grids = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'class_weight': ['balanced', None]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l2'],
            'class_weight': ['balanced', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.1, 0.2],
            'max_depth': [3, 5],
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear'],
            'class_weight': ['balanced', None]
        }
    }
}

# Use F1 score as the scoring metric (good for imbalanced data)
scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"GridSearch setup: {len(param_grids)} models, F1 scoring, 5-fold CV")

# Train models using GridSearchCV
print(f"\n🚀 Training Models with GridSearchCV...")

grid_search_results = {}
best_overall_score = -1
best_overall_model = None
best_model_name = None

for model_name, config in param_grids.items():
    print(f"\nTraining {model_name}...")
    

    # Create GridSearchCV
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        scoring=scorer,
        cv=cv_folds,
        n_jobs=-1,  # Use all available cores
        verbose=0
    )
    
    # Fit GridSearchCV
    grid_search.fit(X_train, y_train)
    
    # Store results
    grid_search_results[model_name] = {
        'grid_search': grid_search,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    }
    
    print(f"  ✅ {model_name}: Best CV F1 = {grid_search.best_score_:.4f}")
    print(f"     Best params: {grid_search.best_params_}")
    
    # Track overall best model
    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        best_overall_model = grid_search.best_estimator_
        best_model_name = model_name
            
print(f"\n🏆 Best Overall Model: {best_model_name} (CV F1: {best_overall_score:.4f})")



🔍 Setting up GridSearchCV...
GridSearch setup: 4 models, F1 scoring, 5-fold CV

🚀 Training Models with GridSearchCV...

Training RandomForest...
  ✅ RandomForest: Best CV F1 = 0.8992
     Best params: {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 50}

Training LogisticRegression...
  ✅ LogisticRegression: Best CV F1 = 0.8057
     Best params: {'C': 10.0, 'class_weight': None, 'penalty': 'l2'}

Training GradientBoosting...


/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty'

  ✅ GradientBoosting: Best CV F1 = 0.8529
     Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}

Training SVM...
  ✅ SVM: Best CV F1 = 0.8603
     Best params: {'C': 10.0, 'class_weight': 'balanced', 'kernel': 'rbf'}

🏆 Best Overall Model: RandomForest (CV F1: 0.8992)


In [16]:
# evaluation
from PyDI.entitymatching import MLBasedMatcher
from PyDI.entitymatching import EmbeddingBlocker
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator

embedding_blocker_aa2a = EmbeddingBlocker(
    df_left=metacritic,
    df_right=dbpedia_games,
    text_cols=['name'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=500,
    output_dir=INPUT_DIR / "blocking-evaluation",
    id_column='id'
)
embedding_candidates_aa2a = embedding_blocker_aa2a.materialize()

# Create MLBasedMatcher and apply trained model
ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_aa2a = ml_matcher.match(
    metacritic, dbpedia_games, candidates=embedding_candidates_aa2a, id_column='id', trained_classifier=best_overall_model
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_aa2a,
    test_pairs=aa2a_test,
    out_dir=INPUT_DIR
)

display(eval_results)

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_aa2a,
    out_dir=INPUT_DIR / "cluster_analysis"
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

{'precision': 0.9322916666666666,
 'recall': 0.9572192513368984,
 'f1': 0.9445910290237467,
 'accuracy': 0.9637305699481865,
 'true_positives': 179,
 'false_positives': 13,
 'false_negatives': 8,
 'true_negatives': 379,
 'threshold_used': 0.0,
 'total_correspondences': 15921,
 'filtered_correspondences': 15921,
 'evaluation_timestamp': '2026-02-01T11:23:41.995461',
 'output_files': ['output/try/score/games4/matching_evaluation_summary.json',
  'output/try/score/games4/matching_detailed_results.csv']}


📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,2476,47.725520
1,3,1192,22.976099
2,4,602,11.603701
3,5,360,6.939090
4,6,180,3.469545
5,7,107,2.062452
6,8,68,1.310717
7,9,54,1.040864
8,10,31,0.597533
9,11,23,0.443331


Testing Company Datasets

In [49]:
from PyDI.entitymatching import FeatureExtractor
import pandas as pd
from pathlib import Path
from PyDI.io import load_csv
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator


INPUT_DIR = Path("output/big/companies1")

fullcontact = pd.read_xml('input/datasets/fullcontact.xml')
forbes = pd.read_csv('input/datasets/forbes.csv')

# Load ground truth correspondences
aa2a_train = load_csv(
    INPUT_DIR / "forbes_fullcontact_goldstandard_blocking.csv",
    name="forbes_fullcontact_goldstandard_blocking",
    header=1,
    names=['id_a', 'id_b', 'label'],
    add_index=False
)

# Load test set and normalize columns to id1/id2
aa2a_test_raw = pd.read_csv(
    "input/datasets/forbes_2_fullcontact_all.csv",
    header=None,
    dtype=str
)
if aa2a_test_raw.shape[1] == 1:
    split_test = aa2a_test_raw.iloc[:, 0].str.split(',', expand=True)
    split_test.columns = ['id_a', 'id_b', 'label']
    aa2a_test = split_test.copy()
else:
    aa2a_test = aa2a_test_raw.copy()
    if aa2a_test.shape[1] >= 3:
        aa2a_test = aa2a_test.iloc[:, :3]
        aa2a_test.columns = ['id_a', 'id_b', 'label']

# Normalize to id1/id2 and clean
aa2a_test = aa2a_test.rename(columns={'id_a': 'id1', 'id_b': 'id2'})
aa2a_test['id1'] = aa2a_test['id1'].astype(str).str.strip()
aa2a_test['id2'] = aa2a_test['id2'].astype(str).str.strip()
aa2a_test['label'] = aa2a_test['label'].astype(str).str.strip().str.upper().map({'TRUE': 1, 'FALSE': 0})
aa2a_test = aa2a_test.dropna(subset=['label'])
aa2a_test['label'] = aa2a_test['label'].astype(int)
aa2a_test = aa2a_test[['id1', 'id2', 'label']]

def normalize_country(val: str) -> str:
    if val is None:
        return ""
    v = str(val).strip().lower()
    v = v.replace(".", "").replace(",", "").replace("-", " ")
    v = " ".join(v.split())
    mapping = {
        "usa": "united states",
        "us": "united states",
        "u s": "united states",
        "u s a": "united states",
        "united states of america": "united states",
        "uk": "united kingdom",
        "u k": "united kingdom",
        "england": "united kingdom",
        "scotland": "united kingdom",
        "wales": "united kingdom",
        "uae": "united arab emirates",
        "korea republic of": "south korea",
        "republic of korea": "south korea",
        "south korea": "south korea",
        "north korea": "north korea",
        "russia": "russian federation",
        "czech republic": "czechia",
        "slovak republic": "slovakia",
        "viet nam": "vietnam",
        "brasil": "brazil",
        "peoples republic of china": "china",
        "prc": "china",
        "hong kong": "china",
        "taiwan": "taiwan",
        "iran": "iran",
        "venezuela": "venezuela",
        "bolivia": "bolivia",
        "tanzania": "tanzania",
        "laos": "laos",
        "myanmar": "myanmar",
        "burma": "myanmar",
        "syria": "syria",
        "ivory coast": "cote d ivoire",
        "cote d'ivoire": "cote d ivoire",
        "cote d ivoire": "cote d ivoire",
        "congo": "congo",
        "democratic republic of congo": "congo",
        "republic of congo": "congo",
        "cape verde": "cabo verde",
        "swaziland": "eswatini",
        "macedonia": "north macedonia",
        "moldova": "moldova",
        "palestine": "palestine",
        "saudi arabia": "saudi arabia",
        # Forbes uses 3-letter codes
        "chn": "china",
        "jpn": "japan",
        "kor": "south korea",
        "deu": "germany",
        "gbr": "united kingdom",
        "fra": "france",
        "can": "canada",
        "aus": "australia",
        "bra": "brazil",
        "ind": "india",
        "rus": "russian federation",
        "ita": "italy",
        "esp": "spain",
        "mex": "mexico",
        "che": "switzerland",
        "nld": "netherlands",
        "swe": "sweden",
        "nor": "norway",
        "dnk": "denmark",
        "fin": "finland",
        "sgp": "singapore",
        "hkg": "china",
        "twn": "taiwan",
        "irl": "ireland",
        "bel": "belgium",
        "aut": "austria",
        "zaf": "south africa",
        "are": "united arab emirates",
        "sau": "saudi arabia",
        "isr": "israel",
        "tha": "thailand",
        "mys": "malaysia",
        "idn": "indonesia",
        "phl": "philippines",
    }
    return mapping.get(v, v)

# IMPROVED similarity comparators - maximizing features for limited shared columns
similarity_comparators = [
    # Name similarity - comprehensive coverage (most important feature!)
    StringComparator("name", similarity_function="jaro_winkler", preprocess=str.lower),
    StringComparator("name", similarity_function="levenshtein", preprocess=str.lower),
    StringComparator("name", similarity_function="cosine", preprocess=str.lower, tokenization="word"),
    StringComparator("name", similarity_function="jaccard", preprocess=str.lower, tokenization="word"),
    StringComparator("name", similarity_function="jaccard", preprocess=str.lower, tokenization="char"),  # char-level
    StringComparator("name", similarity_function="cosine", preprocess=str.lower, tokenization="char"),   # char-level cosine
    
    # Country similarity - normalized with multiple functions
    StringComparator("country", similarity_function="jaccard", preprocess=normalize_country, list_strategy='concatenate'),
    StringComparator("country", similarity_function="jaccard", preprocess=normalize_country, list_strategy='best_match'),
    StringComparator("country", similarity_function="levenshtein", preprocess=normalize_country, list_strategy='best_match'),
    StringComparator("country", similarity_function="jaro_winkler", preprocess=normalize_country, list_strategy='best_match'),
]

feature_extractor = FeatureExtractor(similarity_comparators)

# Extract features using FeatureExtractor
# Rename columns to match FeatureExtractor's expected format (id1, id2)
pairs_df = aa2a_train[['id_a', 'id_b']].rename(columns={'id_a': 'id1', 'id_b': 'id2'})
train_features = feature_extractor.create_features(
    forbes, fullcontact, pairs_df, labels=aa2a_train['label'], id_column='id'
)

print(f"✅ Training features extracted!")
print(f"Feature columns: {[col for col in train_features.columns if col not in ['id1', 'id2', 'label']]}")

# Prepare data for ML training
feature_columns = [col for col in train_features.columns if col not in ['id1', 'id2', 'label']]

X_train = train_features[feature_columns]
y_train = train_features['label']

print(f"Training data: X={X_train.shape}, y={y_train.shape}")
print(f"Class distribution: {y_train.value_counts().to_dict()}")

✅ Training features extracted!
Feature columns: ['StringComparator(name, jaro_winkler, tokenization=char, list_strategy=None)', 'StringComparator(name, levenshtein, tokenization=char, list_strategy=None)', 'StringComparator(name, cosine, tokenization=word, list_strategy=None)', 'StringComparator(name, jaccard, tokenization=word, list_strategy=None)', 'StringComparator(name, jaccard, tokenization=char, list_strategy=None)', 'StringComparator(name, cosine, tokenization=char, list_strategy=None)', 'StringComparator(country, jaccard, tokenization=word, list_strategy=concatenate)', 'StringComparator(country, jaccard, tokenization=word, list_strategy=best_match)', 'StringComparator(country, levenshtein, tokenization=char, list_strategy=best_match)', 'StringComparator(country, jaro_winkler, tokenization=char, list_strategy=best_match)']
Training data: X=(299, 10), y=(299,)
Class distribution: {0: 150, 1: 149}


In [50]:
# Set up GridSearchCV with multiple models and hyperparameters
print(f"\n🔍 Setting up GridSearchCV...")

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score

# Define models and parameter grids
param_grids = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'class_weight': ['balanced', None]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l2'],
            'class_weight': ['balanced', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.1, 0.2],
            'max_depth': [3, 5],
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear'],
            'class_weight': ['balanced', None]
        }
    }
}

# Use F1 score as the scoring metric (good for imbalanced data)
scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"GridSearch setup: {len(param_grids)} models, F1 scoring, 5-fold CV")

# Train models using GridSearchCV
print(f"\n🚀 Training Models with GridSearchCV...")

grid_search_results = {}
best_overall_score = -1
best_overall_model = None
best_model_name = None

for model_name, config in param_grids.items():
    print(f"\nTraining {model_name}...")
    

    # Create GridSearchCV
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        scoring=scorer,
        cv=cv_folds,
        n_jobs=-1,  # Use all available cores
        verbose=0
    )
    
    # Fit GridSearchCV
    grid_search.fit(X_train, y_train)
    
    # Store results
    grid_search_results[model_name] = {
        'grid_search': grid_search,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    }
    
    print(f"  ✅ {model_name}: Best CV F1 = {grid_search.best_score_:.4f}")
    print(f"     Best params: {grid_search.best_params_}")
    
    # Track overall best model
    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        best_overall_model = grid_search.best_estimator_
        best_model_name = model_name
            
print(f"\n🏆 Best Overall Model: {best_model_name} (CV F1: {best_overall_score:.4f})")



🔍 Setting up GridSearchCV...
GridSearch setup: 4 models, F1 scoring, 5-fold CV

🚀 Training Models with GridSearchCV...

Training RandomForest...
  ✅ RandomForest: Best CV F1 = 0.8475
     Best params: {'class_weight': None, 'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}

Training LogisticRegression...
  ✅ LogisticRegression: Best CV F1 = 0.8207
     Best params: {'C': 1.0, 'class_weight': 'balanced', 'penalty': 'l2'}

Training GradientBoosting...


/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty'

  ✅ GradientBoosting: Best CV F1 = 0.8285
     Best params: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 50}

Training SVM...
  ✅ SVM: Best CV F1 = 0.8122
     Best params: {'C': 1.0, 'class_weight': 'balanced', 'kernel': 'linear'}

🏆 Best Overall Model: RandomForest (CV F1: 0.8475)


In [51]:
# evaluation
from PyDI.entitymatching import MLBasedMatcher
from PyDI.entitymatching import EmbeddingBlocker
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator

embedding_blocker_aa2a = EmbeddingBlocker(
    df_left=forbes,
    df_right=fullcontact,
    text_cols=['name'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=200,  # Increased from 150 for better recall
    batch_size=500,
    output_dir=INPUT_DIR / "blocking-evaluation",
    id_column='id'
)
embedding_candidates_aa2a = embedding_blocker_aa2a.materialize()

# Create MLBasedMatcher and apply trained model
ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_aa2a = ml_matcher.match(
    forbes, fullcontact, candidates=embedding_candidates_aa2a, id_column='id', trained_classifier=best_overall_model
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_aa2a,
    test_pairs=aa2a_test,
    out_dir=INPUT_DIR
)

display(eval_results)

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_aa2a,
    out_dir=INPUT_DIR / "cluster_analysis"
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

{'precision': 0.7813267813267813,
 'recall': 0.7473560517038778,
 'f1': 0.763963963963964,
 'accuracy': 0.8544983339503888,
 'true_positives': 636,
 'false_positives': 178,
 'false_negatives': 215,
 'true_negatives': 1672,
 'threshold_used': 0.0,
 'total_correspondences': 158851,
 'filtered_correspondences': 158851,
 'evaluation_timestamp': '2026-02-01T22:53:47.114316',
 'output_files': ['output/big/companies1/matching_evaluation_summary.json',
  'output/big/companies1/matching_detailed_results.csv']}


📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,3913,1,100.0


In [52]:
# Show mismatches with record content
import pandas as pd

detailed = pd.read_csv(INPUT_DIR / "matching_detailed_results.csv")

# Debug classification distribution
print("classification counts:")
print(detailed["classification"].value_counts(dropna=False))

false_positives = detailed[detailed["classification"] == "FP"].copy()
false_negatives = detailed[detailed["classification"] == "FN"].copy()

print(f"FP count: {len(false_positives)} | FN count (from detailed): {len(false_negatives)}")

# Build lookup dicts
forbes_dict = forbes.set_index('id').to_dict('index')
fullcontact_dict = fullcontact.set_index('id').to_dict('index')

def summarize_record(record: dict, preferred_cols: list, max_cols: int = 6):
    if record is None:
        return "NOT FOUND"
    cols = [c for c in preferred_cols if c in record]
    if not cols:
        cols = list(record.keys())[:max_cols]
    parts = []
    for c in cols:
        val = record.get(c)
        if pd.notna(val):
            parts.append(f"{c}: {str(val)[:80]}")
    return " | ".join(parts) if parts else "(no fields)"

print("\n=== FALSE POSITIVES (model predicted match, but not in test set) ===")
if len(false_positives) == 0:
    print("(none)")
for _, row in false_positives.head(10).iterrows():
    id1 = row['id1']
    id2 = row['id2']
    score = row.get('score', None)
    print(f"\nPair: {id1}  <->  {id2}  | score={score}")
    print("  FORBES:", summarize_record(forbes_dict.get(id1), ['name', 'country', 'industry', 'Sector']))
    print("  FULLCONTACT:", summarize_record(fullcontact_dict.get(id2), ['name', 'country', 'industry', 'locale', 'type']))

# If detailed results don't contain FNs, derive them from test pairs vs correspondences
print("\n=== FALSE NEGATIVES (missed true matches) ===")
test_pos = aa2a_test[aa2a_test['label'] == 1][['id1', 'id2']].copy()
pred_pairs = correspondences_aa2a[['id1', 'id2']].copy()

test_set = set(map(tuple, test_pos.values))
pred_set = set(map(tuple, pred_pairs.values))
fn_pairs = [p for p in test_set if p not in pred_set]

if len(false_negatives) == 0:
    print(f"(none in detailed; computed {len(fn_pairs)} from test pairs)")
    for id1, id2 in fn_pairs[:10]:
        print(f"\nPair: {id1}  <->  {id2}")
        print("  FORBES:", summarize_record(forbes_dict.get(id1), ['name', 'country', 'industry', 'Sector']))
        print("  FULLCONTACT:", summarize_record(fullcontact_dict.get(id2), ['name', 'country', 'industry', 'locale', 'type']))
else:
    for _, row in false_negatives.head(10).iterrows():
        id1 = row['id1']
        id2 = row['id2']
        score = row.get('score', None)
        print(f"\nPair: {id1}  <->  {id2}  | score={score}")
        print("  FORBES:", summarize_record(forbes_dict.get(id1), ['name', 'country', 'industry', 'Sector']))
        print("  FULLCONTACT:", summarize_record(fullcontact_dict.get(id2), ['name', 'country', 'industry', 'locale', 'type']))

classification counts:
classification
NaN    158037
TP        636
FP        178
Name: count, dtype: int64
FP count: 178 | FN count (from detailed): 0

=== FALSE POSITIVES (model predicted match, but not in test set) ===

Pair: http://www.forbes.com/companies/bank-of-america/  <->  fullcontact_238  | score=1.0
  FORBES: name: Bank of America | country: USA | industry: Major Banks | Sector: Financials
  FULLCONTACT: name: Crédit Agricole | country: France

Pair: http://www.forbes.com/companies/apple/  <->  fullcontact_1451  | score=1.0
  FORBES: name: Apple | country: USA | industry: Computer Hardware | Sector: Information Technology
  FULLCONTACT: name: ARMANI

Pair: http://www.forbes.com/companies/citigroup/  <->  fullcontact_220  | score=1.0
  FORBES: name: Citigroup | country: USA | industry: Major Banks | Sector: Financials
  FULLCONTACT: name: Clorox Company (the) | country: United States

Pair: http://www.forbes.com/companies/citigroup/  <->  fullcontact_82  | score=1.0
  FORBES: 

In [53]:
from PyDI.entitymatching import FeatureExtractor
import pandas as pd
from pathlib import Path
from PyDI.io import load_csv
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator


INPUT_DIR = Path("output/big/companies1")

fullcontact = pd.read_xml('input/datasets/fullcontact.xml')
forbes = pd.read_csv('input/datasets/forbes.csv')
dbpedia = pd.read_xml('input/datasets/dbpedia.xml')

# Load ground truth correspondences
aa2a_train = load_csv(
    INPUT_DIR / "forbes_dpedia_goldstandard_blocking.csv",
    name="forbes_dbpedia_goldstandard_blocking",
    header=1,
    names=['id_a', 'id_b', 'label'],
    add_index=False
)

# Load test set and normalize columns to id1/id2
aa2a_test_raw = pd.read_csv(
    "input/datasets/forbes_2_dbpedia_test.csv",
    header=None,
    dtype=str
)
if aa2a_test_raw.shape[1] == 1:
    split_test = aa2a_test_raw.iloc[:, 0].str.split(',', expand=True)
    split_test.columns = ['id_a', 'id_b', 'label']
    aa2a_test = split_test.copy()
else:
    aa2a_test = aa2a_test_raw.copy()
    if aa2a_test.shape[1] >= 3:
        aa2a_test = aa2a_test.iloc[:, :3]
        aa2a_test.columns = ['id_a', 'id_b', 'label']

# Normalize to id1/id2 and clean
aa2a_test = aa2a_test.rename(columns={'id_a': 'id1', 'id_b': 'id2'})
aa2a_test['id1'] = aa2a_test['id1'].astype(str).str.strip()
aa2a_test['id2'] = aa2a_test['id2'].astype(str).str.strip()
aa2a_test['label'] = aa2a_test['label'].astype(str).str.strip().str.upper().map({'TRUE': 1, 'FALSE': 0})
aa2a_test = aa2a_test.dropna(subset=['label'])
aa2a_test['label'] = aa2a_test['label'].astype(int)
aa2a_test = aa2a_test[['id1', 'id2', 'label']]

print(f"Train: {len(aa2a_train)} pairs | Test: {len(aa2a_test)} pairs")
print(f"Test matches: {aa2a_test['label'].sum()} | Test non-matches: {(aa2a_test['label'] == 0).sum()}")

def normalize_country(val: str) -> str:
    if val is None:
        return ""
    v = str(val).strip().lower()
    v = v.replace(".", "").replace(",", "").replace("-", " ")
    v = " ".join(v.split())
    mapping = {
        "usa": "united states",
        "us": "united states",
        "u s": "united states",
        "u s a": "united states",
        "united states of america": "united states",
        "uk": "united kingdom",
        "u k": "united kingdom",
        "england": "united kingdom",
        "scotland": "united kingdom",
        "wales": "united kingdom",
        "uae": "united arab emirates",
        "korea republic of": "south korea",
        "republic of korea": "south korea",
        "south korea": "south korea",
        "north korea": "north korea",
        "russia": "russian federation",
        "czech republic": "czechia",
        "slovak republic": "slovakia",
        "viet nam": "vietnam",
        "brasil": "brazil",
        "peoples republic of china": "china",
        "prc": "china",
        "hong kong": "china",
        "taiwan": "taiwan",
        "iran": "iran",
        "venezuela": "venezuela",
        "bolivia": "bolivia",
        "tanzania": "tanzania",
        "laos": "laos",
        "myanmar": "myanmar",
        "burma": "myanmar",
        "syria": "syria",
        "ivory coast": "cote d ivoire",
        "cote d'ivoire": "cote d ivoire",
        "cote d ivoire": "cote d ivoire",
        "congo": "congo",
        "democratic republic of congo": "congo",
        "republic of congo": "congo",
        "cape verde": "cabo verde",
        "swaziland": "eswatini",
        "macedonia": "north macedonia",
        "moldova": "moldova",
        "palestine": "palestine",
        "saudi arabia": "saudi arabia",
        # Forbes uses 3-letter codes
        "chn": "china",
        "jpn": "japan",
        "kor": "south korea",
        "deu": "germany",
        "gbr": "united kingdom",
        "fra": "france",
        "can": "canada",
        "aus": "australia",
        "bra": "brazil",
        "ind": "india",
        "rus": "russian federation",
        "ita": "italy",
        "esp": "spain",
        "mex": "mexico",
        "che": "switzerland",
        "nld": "netherlands",
        "swe": "sweden",
        "nor": "norway",
        "dnk": "denmark",
        "fin": "finland",
        "sgp": "singapore",
        "hkg": "china",
        "twn": "taiwan",
        "irl": "ireland",
        "bel": "belgium",
        "aut": "austria",
        "zaf": "south africa",
        "are": "united arab emirates",
        "sau": "saudi arabia",
        "isr": "israel",
        "tha": "thailand",
        "mys": "malaysia",
        "idn": "indonesia",
        "phl": "philippines",
        "portugal": "portugal",
    }
    return mapping.get(v, v)

# Only use columns that reliably exist in both datasets: name, country
similarity_comparators = [
    # Name similarity - comprehensive coverage (most important feature!)
    StringComparator("name", similarity_function="jaro_winkler", preprocess=str.lower),
    StringComparator("name", similarity_function="levenshtein", preprocess=str.lower),
    StringComparator("name", similarity_function="cosine", preprocess=str.lower, tokenization="word"),
    StringComparator("name", similarity_function="jaccard", preprocess=str.lower, tokenization="word"),
    StringComparator("name", similarity_function="jaccard", preprocess=str.lower, tokenization="char"),
    StringComparator("name", similarity_function="cosine", preprocess=str.lower, tokenization="char"),
    
    # Country similarity - normalized with multiple functions
    StringComparator("country", similarity_function="jaccard", preprocess=normalize_country),
    StringComparator("country", similarity_function="levenshtein", preprocess=normalize_country),
    StringComparator("country", similarity_function="jaro_winkler", preprocess=normalize_country),
]

feature_extractor = FeatureExtractor(similarity_comparators)

# Extract features using FeatureExtractor
pairs_df = aa2a_train[['id_a', 'id_b']].rename(columns={'id_a': 'id1', 'id_b': 'id2'})
train_features = feature_extractor.create_features(
    forbes, dbpedia, pairs_df, labels=aa2a_train['label'], id_column='id'
)

print(f"✅ Training features extracted!")
print(f"Feature columns: {[col for col in train_features.columns if col not in ['id1', 'id2', 'label']]}")

# Prepare data for ML training
feature_columns = [col for col in train_features.columns if col not in ['id1', 'id2', 'label']]

X_train = train_features[feature_columns]
y_train = train_features['label']

print(f"Training data: X={X_train.shape}, y={y_train.shape}")
print(f"Class distribution: {y_train.value_counts().to_dict()}")

Train: 299 pairs | Test: 140 pairs
Test matches: 69 | Test non-matches: 71
✅ Training features extracted!
Feature columns: ['StringComparator(name, jaro_winkler, tokenization=char, list_strategy=None)', 'StringComparator(name, levenshtein, tokenization=char, list_strategy=None)', 'StringComparator(name, cosine, tokenization=word, list_strategy=None)', 'StringComparator(name, jaccard, tokenization=word, list_strategy=None)', 'StringComparator(name, jaccard, tokenization=char, list_strategy=None)', 'StringComparator(name, cosine, tokenization=char, list_strategy=None)', 'StringComparator(country, jaccard, tokenization=word, list_strategy=None)', 'StringComparator(country, levenshtein, tokenization=char, list_strategy=None)', 'StringComparator(country, jaro_winkler, tokenization=char, list_strategy=None)']
Training data: X=(299, 9), y=(299,)
Class distribution: {0: 150, 1: 149}


In [54]:
# Set up GridSearchCV with multiple models and hyperparameters
print(f"\n🔍 Setting up GridSearchCV...")

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score

# Define models and parameter grids
param_grids = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'class_weight': ['balanced', None]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l2'],
            'class_weight': ['balanced', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.1, 0.2],
            'max_depth': [3, 5],
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear'],
            'class_weight': ['balanced', None]
        }
    }
}

# Use F1 score as the scoring metric (good for imbalanced data)
scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"GridSearch setup: {len(param_grids)} models, F1 scoring, 5-fold CV")

# Train models using GridSearchCV
print(f"\n🚀 Training Models with GridSearchCV...")

grid_search_results = {}
best_overall_score = -1
best_overall_model = None
best_model_name = None

for model_name, config in param_grids.items():
    print(f"\nTraining {model_name}...")
    

    # Create GridSearchCV
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        scoring=scorer,
        cv=cv_folds,
        n_jobs=-1,  # Use all available cores
        verbose=0
    )
    
    # Fit GridSearchCV
    grid_search.fit(X_train, y_train)
    
    # Store results
    grid_search_results[model_name] = {
        'grid_search': grid_search,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    }
    
    print(f"  ✅ {model_name}: Best CV F1 = {grid_search.best_score_:.4f}")
    print(f"     Best params: {grid_search.best_params_}")
    
    # Track overall best model
    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        best_overall_model = grid_search.best_estimator_
        best_model_name = model_name
            
print(f"\n🏆 Best Overall Model: {best_model_name} (CV F1: {best_overall_score:.4f})")



🔍 Setting up GridSearchCV...
GridSearch setup: 4 models, F1 scoring, 5-fold CV

🚀 Training Models with GridSearchCV...

Training RandomForest...
  ✅ RandomForest: Best CV F1 = 0.9800
     Best params: {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}

Training LogisticRegression...
  ✅ LogisticRegression: Best CV F1 = 0.9800
     Best params: {'C': 10.0, 'class_weight': 'balanced', 'penalty': 'l2'}

Training GradientBoosting...


/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nickblazek/Desktop/Agent_Pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty'

  ✅ GradientBoosting: Best CV F1 = 0.9800
     Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}

Training SVM...
  ✅ SVM: Best CV F1 = 0.9833
     Best params: {'C': 1.0, 'class_weight': 'balanced', 'kernel': 'rbf'}

🏆 Best Overall Model: SVM (CV F1: 0.9833)


In [55]:
# evaluation - Forbes ↔ DBpedia
from PyDI.entitymatching import MLBasedMatcher
from PyDI.entitymatching import EmbeddingBlocker
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator

embedding_blocker_aa2a = EmbeddingBlocker(
    df_left=forbes,
    df_right=dbpedia,
    text_cols=['name'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=500,
    batch_size=500,
    output_dir=INPUT_DIR / "blocking-evaluation",
    id_column='id'
)
embedding_candidates_aa2a = embedding_blocker_aa2a.materialize()

# Create MLBasedMatcher
ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_aa2a = ml_matcher.match(
    forbes, dbpedia, candidates=embedding_candidates_aa2a, id_column='id', 
    trained_classifier=best_overall_model, threshold=0.3  # Lower threshold to improve recall
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_aa2a,
    test_pairs=aa2a_test,
    out_dir=INPUT_DIR
)

display(eval_results)

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_aa2a,
    out_dir=INPUT_DIR / "cluster_analysis"
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

{'precision': 1.0,
 'recall': 0.5797101449275363,
 'f1': 0.7339449541284404,
 'accuracy': 0.7928571428571428,
 'true_positives': 40,
 'false_positives': 0,
 'false_negatives': 29,
 'true_negatives': 71,
 'threshold_used': 0.0,
 'total_correspondences': 232,
 'filtered_correspondences': 232,
 'evaluation_timestamp': '2026-02-01T22:56:45.375677',
 'output_files': ['output/big/companies1/matching_evaluation_summary.json',
  'output/big/companies1/matching_detailed_results.csv']}


📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,232,100.0


In [56]:
# Analyze False Negatives - what matches are we missing?
import pandas as pd

# Get true positive pairs from test set
test_positives = aa2a_test[aa2a_test['label'] == 1][['id1', 'id2']].copy()
pred_pairs = correspondences_aa2a[['id1', 'id2']].copy()

# Find false negatives (in test but not predicted)
test_set = set(map(tuple, test_positives.values))
pred_set = set(map(tuple, pred_pairs.values))
fn_pairs = [p for p in test_set if p not in pred_set]

print(f"=== FALSE NEGATIVES ANALYSIS ({len(fn_pairs)} missed matches) ===\n")

# Build lookup dicts
forbes_dict = forbes.set_index('id').to_dict('index')
dbpedia_dict = dbpedia.set_index('id').to_dict('index')

# Check if FN pairs are even in the blocking candidates
blocking_set = set(map(tuple, embedding_candidates_aa2a[['id1', 'id2']].values))
fn_in_blocking = sum(1 for p in fn_pairs if p in blocking_set)
print(f"FN pairs found in blocking candidates: {fn_in_blocking}/{len(fn_pairs)}")
print(f"FN pairs NOT in blocking (blocking recall issue): {len(fn_pairs) - fn_in_blocking}\n")

# Show the missed matches
for i, (id1, id2) in enumerate(fn_pairs[:15]):
    in_blocking = "✓ in candidates" if (id1, id2) in blocking_set else "✗ NOT in candidates"
    
    forbes_rec = forbes_dict.get(id1, {})
    dbpedia_rec = dbpedia_dict.get(id2, {})
    
    f_name = forbes_rec.get('name', 'N/A')
    f_country = forbes_rec.get('country', 'N/A')
    d_name = dbpedia_rec.get('name', 'N/A')
    d_country = dbpedia_rec.get('country', 'N/A')
    
    print(f"--- Missed Match #{i+1} [{in_blocking}] ---")
    print(f"  FORBES:  name='{f_name}' | country={f_country}")
    print(f"  DBPEDIA: name='{d_name}' | country={d_country}")
    print()

=== FALSE NEGATIVES ANALYSIS (29 missed matches) ===



ValueError: DataFrame index must be unique for orient='index'.